# Estadística descriptiva a gran escala con Python
### Big Data · Unidad 6 · Ejercicio introductorio

**Caso: DomiYa** (app ficticia de domicilios en Colombia). En 2025 registró **10,2 millones de pedidos** en seis ciudades. La gerencia quiere entender cuánto compran los clientes, qué tan variable es ese gasto, cuánto tardan las entregas y si hay datos anómalos.

**Herramientas modernas que usaremos**
| Herramienta | Para qué |
|---|---|
| **Polars** | DataFrames rápidos y *lazy*: calcula sin cargar todo en memoria |
| **DuckDB** | SQL analítico directo sobre archivos Parquet |
| **Parquet** | Formato columnar comprimido, estándar en Big Data |

**Cómo trabajar:** ejecuten las celdas **en orden** con **Shift + Enter**. Después de cada resultado lean *Qué observar*.

## Paso 0 · Instalar e importar (≈ 30 segundos)

In [ ]:
%pip install -q -U polars duckdb
import polars as pl, duckdb, numpy as np, matplotlib.pyplot as plt, os, glob, time
pl.Config.set_tbl_rows(20); pl.Config.set_thousands_separator(".") ; pl.Config.set_decimal_separator(",")
fmt = lambda x, d=0: f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")   # formato colombiano
print("✅ Polars", pl.__version__, "| DuckDB", duckdb.__version__)

## Paso 1 · Generar el dataset: 10,2 millones de pedidos (≈ 20 segundos)
Se crean **12 archivos Parquet** (uno por mes), como los guardaría un Data Lake. Los datos son sintéticos pero siguen distribuciones realistas.

| Columna | Descripción |
|---|---|
| `valor_cop` | Valor del pedido en pesos |
| `distancia_km` | Distancia de la entrega |
| `tiempo_min` | Minutos desde el pedido hasta la entrega |
| `calificacion` | Estrellas que da el cliente (1 a 5) |
| `propina_cop` | Propina en pesos (muchos clientes no dan) |
| `ciudad`, `categoria`, `mes`, `fecha_hora` | Variables para agrupar |

In [ ]:
rng = np.random.default_rng(2026)
os.makedirs("pedidos", exist_ok=True)
CIUDADES = np.array(["Bogotá", "Medellín", "Cali", "Barranquilla", "Bucaramanga", "Tunja"])
P_CIU    = [0.40, 0.20, 0.15, 0.12, 0.08, 0.05]
EXTRA_MIN = np.array([8, 4, 3, 2, 0, -4])                     # efecto del tráfico por ciudad
CATS     = np.array(["Restaurantes", "Mercado", "Farmacia", "Licores"])
P_CAT    = [0.55, 0.25, 0.12, 0.08]
MEDIANA_CAT = np.array([32000, 85000, 45000, 70000])
TEMPORADA = [0.95, 0.93, 0.97, 1.0, 1.02, 1.0, 1.03, 1.0, 0.98, 1.0, 1.08, 1.30]   # diciembre gasta más
N = 850_000                                                   # pedidos por mes

t0 = time.time()
for mes in range(1, 13):
    cat = rng.choice(4, N, p=P_CAT); ciu = rng.choice(6, N, p=P_CIU)
    valor = np.round(MEDIANA_CAT[cat] * TEMPORADA[mes-1] * rng.lognormal(0, 0.55, N), -2)
    dist = np.round(rng.gamma(2.2, 1.6, N), 2)
    tiempo = np.round(12 + 4.5 * dist + EXTRA_MIN[ciu] + rng.gamma(2, 5, N)).astype(np.int64)
    calif = rng.choice([1, 2, 3, 4, 5], N, p=[0.04, 0.04, 0.10, 0.27, 0.55])
    propina = np.where(rng.random(N) < 0.6, 0, np.round(rng.lognormal(8.0, 0.5, N), -2))
    i = rng.choice(N, 80, replace=False);  valor[i] *= 25                  # errores de digitación
    i = rng.choice(N, 120, replace=False); tiempo[i] += rng.integers(150, 400, 120)   # pedidos "olvidados"
    fecha = (np.datetime64(f"2025-{mes:02d}-01T00:00:00")
             + rng.integers(0, 28, N).astype("timedelta64[D]")
             + rng.integers(0, 86400, N).astype("timedelta64[s]")).astype("datetime64[ms]")
    pl.DataFrame({"id_pedido": np.arange(N) + mes * 10_000_000, "fecha_hora": fecha, "mes": mes,
                  "ciudad": CIUDADES[ciu], "categoria": CATS[cat], "valor_cop": valor.astype(np.int64),
                  "distancia_km": dist, "tiempo_min": tiempo, "calificacion": calif,
                  "propina_cop": propina.astype(np.int64)}).write_parquet(f"pedidos/pedidos_2025_{mes:02d}.parquet")

mb = sum(os.path.getsize(f) for f in glob.glob("pedidos/*.parquet")) / 1e6
print(f"✅ 12 archivos generados en {time.time()-t0:.0f} s · {mb:.0f} MB en disco")

**Qué observar:** ~10 millones de filas ocupan apenas ~115 MB gracias a **Parquet**, que comprime por columna. Abran el panel **Archivos** (ícono de carpeta) y verán la carpeta `pedidos/`.

## Paso 2 · Conocer el dataset **sin cargarlo**
`pl.scan_parquet` no lee los datos: crea un **plan perezoso** (*lazy*). Polars solo lee lo necesario cuando le pedimos un resultado con `.collect()`. Esto es clave en Big Data: **el dato puede ser más grande que la memoria**.

In [ ]:
lf = pl.scan_parquet("pedidos/*.parquet")          # LazyFrame: todavía no lee nada
print("Columnas y tipos:", lf.collect_schema())
print("Filas totales:", fmt(lf.select(pl.len()).collect().item()))
lf.head(5).collect()

**Qué observar:** contar 10,2 millones de filas es casi instantáneo: Parquet guarda el número de filas en sus metadatos, así que no necesita leer los datos.

## Paso 3 · Tendencia central: ¿cuál es el pedido «típico»?
| Medida | Fórmula / idea | Sensible a atípicos |
|---|---|---|
| **Media** | $\bar{x}=\frac{1}{n}\sum x_i$ | Sí, mucho |
| **Mediana** | El valor del medio al ordenar | No |
| **Moda** | El valor más frecuente | No (útil en categorías y discretos) |
| **Media recortada 5 %** | Media sin el 5 % más bajo ni el 5 % más alto | Poco |

In [ ]:
v = pl.col("valor_cop")
p5, p95 = lf.select(v.quantile(0.05), v.quantile(0.95).alias("p95")).collect().row(0)

central = lf.select(
    media   = v.mean(),
    mediana = v.median(),
    media_recortada_5 = v.filter(v.is_between(p5, p95)).mean(),
).collect(engine="streaming")
print("VALOR DEL PEDIDO (COP)"); display(central)

print("MODAS"); display(lf.select(pl.col("calificacion").mode(), pl.col("ciudad").mode(), pl.col("categoria").mode()).collect())
print("CALIFICACIÓN: media vs mediana"); display(lf.select(media=pl.col("calificacion").mean(), mediana=pl.col("calificacion").median()).collect())

**Qué observar**
- La **media (~$59.400) es mayor que la mediana ($45.000)**: unos pocos pedidos muy caros «jalan» la media hacia arriba. Esto indica una distribución **asimétrica a la derecha**.
- **Interpretación para la gerencia:** «el pedido típico es de unos $45.000» (mediana) es más honesto que «el pedido promedio es de $59.400».
- `engine="streaming"` procesa los datos por bloques: así se calcula aunque no quepan en memoria.

## Paso 4 · Dispersión: ¿qué tanto varían los pedidos?
| Medida | Idea | Robusta |
|---|---|---|
| **Rango** | máx − mín | No |
| **Varianza** $s^2$ | Promedio de las distancias al cuadrado a la media | No |
| **Desviación estándar** $s$ | $\sqrt{s^2}$, en las mismas unidades del dato | No |
| **Rango intercuartílico (IQR)** | $Q_3 - Q_1$: el 50 % central de los datos | Sí |
| **Coeficiente de variación (CV)** | $s / \bar{x}$: dispersión relativa, sin unidades | No |
| **MAD** | Mediana de $\lvert x_i - \text{mediana}\rvert$ | Sí |

In [ ]:
dispersion = lf.select(
    minimo = v.min(), maximo = v.max(), rango = v.max() - v.min(),
    desv_estandar = v.std(),
    q1 = v.quantile(0.25), q3 = v.quantile(0.75),
    iqr = v.quantile(0.75) - v.quantile(0.25),
    cv_porcentaje = v.std() / v.mean() * 100,
    mad = (v - v.median()).abs().median(),
).collect(engine="streaming")
dispersion

**Qué observar**
- El **rango** es enorme (de $2.000 a más de $7 millones) porque depende solo de los dos valores extremos: casi no sirve para describir.
- La **desviación estándar** (~$51.000) está inflada por los atípicos. El **IQR** (~$46.000) y la **MAD** (~$20.000) son **robustas**.
- **CV ≈ 86 %**: el gasto es muy heterogéneo. Un CV por encima de 30 % suele considerarse alta dispersión.

## Paso 5 · Forma de la distribución: percentiles, asimetría y curtosis
- **Percentil $p$:** el valor por debajo del cual está el $p$ % de los pedidos.
- **Asimetría (*skewness*):** 0 = simétrica; > 0 = cola larga a la derecha; < 0 = cola a la izquierda.
- **Curtosis (exceso):** 0 = colas como la normal; > 0 = colas pesadas (más valores extremos de lo esperado).

In [ ]:
percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 0.999]
display(lf.select([v.quantile(p).alias(f"p{p*100:g}") for p in percentiles]).collect(engine="streaming"))

forma = lf.select(asimetria = v.skew(), curtosis = v.kurtosis()).collect(engine="streaming")
forma

**Qué observar**
- Del **p50 al p99** el valor se multiplica por más de 5: la cola derecha es larga.
- **Asimetría ≈ 10** y **curtosis ≈ 660**: valores altísimos. ¿Es la forma natural de los datos o hay algo raro? Lo resolvemos en el Paso 7.

## Paso 6 · Visualizar la distribución sin traer 10 millones de puntos
Con Big Data **no se grafican los datos crudos**: se **agregan primero** (conteos por intervalo) y se grafica el resumen. Polars cuenta en bloques de $5.000 y solo trae ~60 filas a memoria.

In [ ]:
ANCHO = 5000
hist = (lf.filter(v <= 300_000)
          .group_by(((v // ANCHO) * ANCHO).alias("intervalo"))
          .agg(pl.len().alias("pedidos"))
          .sort("intervalo")
          .collect(engine="streaming"))
med, media = central["mediana"][0], central["media"][0]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].bar(hist["intervalo"], hist["pedidos"], width=ANCHO, color="#2F7DE1", align="edge")
ax[0].axvline(med, color="#12366B", lw=2.5, label=f"Mediana ${fmt(med)}")
ax[0].axvline(media, color="#E08A1E", lw=2.5, ls="--", label=f"Media ${fmt(media)}")
ax[0].set_title("Valor del pedido (hasta $300.000)"); ax[0].set_xlabel("COP"); ax[0].set_ylabel("Pedidos"); ax[0].legend()

cal = lf.group_by("calificacion").agg(pl.len().alias("n")).sort("calificacion").collect()
ax[1].bar(cal["calificacion"], cal["n"], color="#12366B")
ax[1].set_title("Calificación (variable discreta)"); ax[1].set_xlabel("Estrellas"); ax[1].set_ylabel("Pedidos")
plt.tight_layout(); plt.show()
print("Filas traídas a memoria para el histograma:", hist.height)

**Qué observar:** el valor tiene la forma típica de una distribución **log-normal**: muchos pedidos baratos y una cola larga de pedidos caros. La calificación está **sesgada a la izquierda**: la moda y la mediana son 5 estrellas.

## Paso 7 · Valores atípicos: ¿error o dato real?
**Regla del IQR (Tukey):** es atípico lo que está por encima de $Q_3 + 1{,}5 \cdot IQR$ o por debajo de $Q_1 - 1{,}5 \cdot IQR$.

In [ ]:
q1, q3 = dispersion["q1"][0], dispersion["q3"][0]
limite = q3 + 1.5 * (q3 - q1)
print(f"Límite superior (Tukey): ${fmt(limite)}")
display(lf.select(atipicos_tukey = (v > limite).sum(), porcentaje = (v > limite).mean() * 100).collect())

print("¿Cuántos pedidos superan $1.000.000?")
display(lf.filter(v > 1_000_000).select(pl.len(), v.min().alias("minimo"), v.max().alias("maximo")).collect())

print("Estadísticos SIN los pedidos de más de $1.000.000:")
display(pl.concat([
    lf.select(pl.lit("todos").alias("datos"), media=v.mean(), mediana=v.median(), desv=v.std(), asimetria=v.skew(), curtosis=v.kurtosis()).collect(),
    lf.filter(v <= 1_000_000).select(pl.lit("sin > $1M").alias("datos"), media=v.mean(), mediana=v.median(), desv=v.std(), asimetria=v.skew(), curtosis=v.kurtosis()).collect()]))

**Qué observar**
- Tukey marca ~**6 %** de pedidos como atípicos. Pero en una distribución con cola larga, **muchos son pedidos reales** (un mercado grande). **Atípico no significa error.**
- Solo **578 pedidos** superan $1.000.000; parecen **errores de digitación** (un cero de más). Al quitarlos, la **mediana no cambia**, pero la curtosis cae de ~660 a ~13 y la asimetría de ~10 a ~2,7.
- **Lección:** la mediana y el IQR son robustos; la media, la desviación, la asimetría y la curtosis son muy sensibles a unos pocos valores.

## Paso 8 · Comparar grupos: el poder de `group_by` a gran escala

In [ ]:
por_categoria = (lf.group_by("categoria")
    .agg(pedidos=pl.len(), media=v.mean(), mediana=v.median(), desv=v.std(),
         cv=(v.std() / v.mean() * 100), p95=v.quantile(0.95))
    .sort("mediana", descending=True).collect(engine="streaming"))
display(por_categoria)

t = pl.col("tiempo_min")
por_ciudad = (lf.group_by("ciudad")
    .agg(pedidos=pl.len(), media_min=t.mean(), mediana_min=t.median(),
         p90_min=t.quantile(0.90), pct_mas_60=(t > 60).mean() * 100)
    .sort("mediana_min", descending=True).collect(engine="streaming"))
display(por_ciudad)

In [ ]:
# Diagrama de cajas construido con estadísticos agregados (sin cargar los 10 millones de valores)
cajas = (lf.group_by("categoria").agg(
            med=v.median(), q1=v.quantile(0.25), q3=v.quantile(0.75),
            whislo=v.quantile(0.05), whishi=v.quantile(0.95), mean=v.mean())
         .sort("med").collect())
stats = [dict(label=r["categoria"], med=r["med"], q1=r["q1"], q3=r["q3"], whislo=r["whislo"],
              whishi=r["whishi"], mean=r["mean"], fliers=[]) for r in cajas.iter_rows(named=True)]
fig, ax = plt.subplots(figsize=(10, 4.5))
import matplotlib
orient = {"orientation": "horizontal"} if tuple(map(int, matplotlib.__version__.split(".")[:2])) >= (3, 10) else {"vert": False}
ax.bxp(stats, showmeans=True, patch_artist=True, **orient,
       boxprops=dict(facecolor="#E3EEFB", edgecolor="#12366B"), medianprops=dict(color="#12366B", lw=2.5))
ax.set_title("Valor del pedido por categoría (bigotes: p5 y p95 · triángulo: media)"); ax.set_xlabel("COP")
plt.tight_layout(); plt.show()

**Qué observar**
- **Mercado** tiene la mediana más alta; **Restaurantes**, la más baja pero el mayor volumen.
- En todas las categorías **media > mediana**: la asimetría se mantiene dentro de cada grupo.
- **Bogotá** tiene las entregas más lentas y **Tunja** las más rápidas. El indicador `pct_mas_60` (% de pedidos de más de una hora) es más útil para la operación que la media.

## Paso 9 · A gran escala I: ¿qué estadísticos se pueden calcular por partes?
En Big Data los datos están **repartidos** en muchos archivos o servidores. Cada parte calcula un resumen (**map**) y luego se combinan (**reduce**).
- **Combinables:** conteo, suma, suma de cuadrados → con ellos se obtienen exactamente la **media** y la **varianza** globales.
- **No combinables:** la **mediana** y los percentiles. La mediana de las medianas **no** es la mediana global.

In [ ]:
partes = []
for f in sorted(glob.glob("pedidos/*.parquet")):           # MAP: un resumen por archivo
    partes.append(pl.scan_parquet(f).select(
        archivo=pl.lit(os.path.basename(f)), n=pl.len(), suma=v.sum(),
        suma_cuadrados=(v.cast(pl.Float64) ** 2).sum(), mediana=v.median()).collect())
partes = pl.concat(partes)
display(partes)

n, s, s2 = partes["n"].sum(), partes["suma"].sum(), partes["suma_cuadrados"].sum()   # REDUCE
media_comb = s / n
desv_comb = ((s2 - n * media_comb**2) / (n - 1)) ** 0.5
exacto = lf.select(media=v.mean(), desv=v.std(), mediana=v.median()).collect()

print(f"Media combinada      : {fmt(media_comb, 2):>12} | exacta: {fmt(exacto['media'][0], 2)}")
print(f"Desv. est. combinada : {fmt(desv_comb, 2):>12} | exacta: {fmt(exacto['desv'][0], 2)}")
print(f"Mediana de medianas  : {fmt(partes['mediana'].median()):>12} | mediana real: {fmt(exacto['mediana'][0])}")

**Qué observar:** la media y la desviación combinadas coinciden **exactamente** con las globales. La mediana de medianas **no coincide** (44.200 frente a 45.000): cada mes tiene una distribución distinta (diciembre gasta mucho más) y las medianas parciales no se pueden «promediar». Por eso los sistemas distribuidos usan **algoritmos aproximados** para la mediana y los percentiles.

## Paso 10 · A gran escala II: exacto vs. aproximado con DuckDB
DuckDB ejecuta SQL directamente sobre los archivos Parquet, sin importarlos.
- `approx_quantile` usa **T-Digest**: percentiles aproximados con muy poca memoria y combinables.
- `approx_count_distinct` usa **HyperLogLog**: cuenta valores distintos con memoria mínima.

In [ ]:
con = duckdb.connect()
def medir(sql):
    t0 = time.time(); r = con.sql(sql).fetchone(); return r, time.time() - t0

consultas = {
    "Mediana y p99 EXACTOS":      "SELECT quantile_cont(valor_cop, 0.5), quantile_cont(valor_cop, 0.99) FROM 'pedidos/*.parquet'",
    "Mediana y p99 APROXIMADOS":  "SELECT approx_quantile(valor_cop, 0.5), approx_quantile(valor_cop, 0.99) FROM 'pedidos/*.parquet'",
    "Pedidos distintos EXACTO":   "SELECT count(DISTINCT id_pedido) FROM 'pedidos/*.parquet'",
    "Pedidos distintos APROX.":   "SELECT approx_count_distinct(id_pedido) FROM 'pedidos/*.parquet'",
}
for nombre, sql in consultas.items():
    r, seg = medir(sql)
    print(f"{nombre:27s} → {r}   ({seg:.2f} s)")

**Qué observar**
- La mediana aproximada difiere del valor exacto en menos de 0,2 %.
- El conteo aproximado de distintos tiene un error de algunos puntos porcentuales, pero usa una fracción mínima de memoria. Con miles de millones de filas, **la diferencia en memoria y tiempo es enorme**.
- **Decisión profesional:** para un reporte contable se usan valores exactos; para un tablero que se actualiza cada minuto, los aproximados son suficientes.

## Paso 11 · Resumen general con `describe()`

In [ ]:
lf.select("valor_cop", "distancia_km", "tiempo_min", "calificacion", "propina_cop").describe()

**Qué observar:** en `propina_cop` la **mediana es 0** pero la **media es ~$1.350**. Más de la mitad de los clientes no da propina. Una sola cifra no basta: hay que reportar también **qué porcentaje da propina** y **cuánto dan quienes sí la dan**.

## 🏆 Reto del equipo (15 minutos)
La gerencia pregunta: **«¿Cómo se comportan las propinas?»** Usando `lf` y Polars, calculen:
1. El **porcentaje** de pedidos con propina mayor que cero.
2. La **media** y la **mediana** de la propina **solo entre quienes sí dan propina**.
3. La mediana de la propina **por ciudad** (solo pedidos con propina).
4. Escriban **dos frases** para la gerencia: ¿qué medida de tendencia central usarían y por qué?

*Pista:* `pl.col("propina_cop").filter(pl.col("propina_cop") > 0).median()`

In [ ]:
p = pl.col("propina_cop")
# Escriban aquí su solución


## Conclusión: ¿qué estadístico usar?
| Situación | Tendencia central | Dispersión |
|---|---|---|
| Datos simétricos y sin atípicos | Media | Desviación estándar |
| Datos asimétricos o con atípicos (dinero, tiempos) | **Mediana** | **IQR o MAD** |
| Datos categóricos o discretos (ciudad, estrellas) | **Moda** | Frecuencias y porcentajes |
| Muchos ceros (propinas) | Reportar % de ceros + mediana de los no ceros | IQR de los no ceros |

**A gran escala:** usar formatos columnares (Parquet), ejecución *lazy* y por bloques (*streaming*), estadísticos combinables y algoritmos aproximados cuando la exactitud total no es necesaria.